In [8]:
from sklearn.linear_model import LogisticRegression
from src.data import load_data, split_by_year
from src.features import add_ratios
from src.config import RISK_TREND
from src.metrics import evaluate
from src.scorecard import WOEBinner
import numpy as np

In [9]:
train, val, test = (add_ratios(d) for d in split_by_year(load_data()))

In [10]:
# 1. bins from train only, then check them
binner = WOEBinner(RISK_TREND).fit(train, train["default"])
binner.iv()

mve_tl    1.326587
tl_ta     1.046757
quick     0.835920
ni_ta     0.768567
re_ta     0.538619
log_ta    0.087278
dtype: float64

In [11]:
W_tr, W_va, W_te = (binner.transform(d) for d in (train, val, test))

In [12]:
lr = LogisticRegression(max_iter = 1000, C = np.inf).fit(W_tr, train["default"])

dict(zip(W_tr.columns, lr.coef_[0]))

{'mve_tl': np.float64(-0.688657526004746),
 'tl_ta': np.float64(-0.11781538053058813),
 'ni_ta': np.float64(-0.5975231566354839),
 'quick': np.float64(-0.34159566908513217),
 're_ta': np.float64(-0.1900185486693017),
 'log_ta': np.float64(0.001435610789239503)}

In [13]:
import pandas as pd
ref = train["default"].mean()  # Brier skill baseline: the historical default rate
pd.DataFrame({
    "train": evaluate(train["default"], lr.predict_proba(W_tr)[:,1], ref),
    "val":evaluate(val["default"], lr.predict_proba(W_va)[:,1], ref),
    "test":evaluate(test["default"], lr.predict_proba(W_te)[:,1], ref),
})

,train,val,test
n,55927.000000,10473.000000,12282.000000
n_pos,403.000000,87.000000,119.000000
base_rate,0.007206,0.008307,0.009689
pr_auc,0.045238,0.076397,0.103143
pr_auc_lift,6.278017,9.196591,10.645406
roc_auc,0.850316,0.909378,0.906274
brier,0.006988,0.007925,0.009159
brier_skill,0.023195,0.038059,0.045405
mean_pred,0.007203,0.006463,0.006898
ks,0.575841,0.684861,0.676738
